In [57]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.metrics import classification_report,accuracy_score,confusion_matrix,f1_score,make_scorer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from collections import Counter
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import optuna
import pickle
import tqdm as notebook_tqdm
import warnings
from imblearn.over_sampling import ADASYN
warnings.filterwarnings('ignore')


In [2]:
model_dt=DecisionTreeClassifier()
model_dt2=DecisionTreeClassifier()
mms=MinMaxScaler()
sc=StandardScaler()
sm=SMOTEENN()
adasyn=ADASYN(random_state=42)
smote=SMOTE(random_state=42)
model_dt_smotenn=DecisionTreeClassifier()
model_rf_smoteenn=RandomForestClassifier(n_estimators=500)
model_xgb=XGBClassifier(random_state=42)
model_xgb_smoteenn=XGBClassifier(random_state=42)
model_xgb_smote=XGBClassifier(random_state=42)
model_rf=RandomForestClassifier(class_weight='balanced',random_state=42,n_estimators=300,max_depth=6)
model_lgb=LGBMClassifier(random_state=42)
model_cat=CatBoostClassifier(auto_class_weights='Balanced',verbose=0,random_state=42)
model_xgb_adasyn=XGBClassifier(random_state=42)
model_ada_weighted=AdaBoostClassifier(n_estimators=50,random_state=42)

In [3]:
df=pd.read_csv('../data/Customer-Churn.csv')

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df.Churn.value_counts()/len(df)*100

Churn
No     73.463013
Yes    26.536987
Name: count, dtype: float64

## **Churn Rate**: 26.53%

- Which means, 26.53% of the customers churn out of this telecom company

In [6]:
y=df['Churn']

x=df.drop(columns=['customerID','Churn'])

print("Shape of x :",x.shape)
print("Shape of y : ",y.shape)

Shape of x : (7043, 19)
Shape of y :  (7043,)


## Train Test Split

In [7]:
x=pd.get_dummies(x,drop_first=True)
y=df['Churn'].map({'No':0,'Yes':1})

In [8]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.1)

## Model Building

In [9]:
model_dt.fit(x_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [10]:
y_pred_dt=model_dt.predict(x_test)

In [11]:
print(classification_report(y_test,y_pred_dt))

              precision    recall  f1-score   support

           0       0.81      0.85      0.83       512
           1       0.54      0.48      0.51       193

    accuracy                           0.75       705
   macro avg       0.68      0.66      0.67       705
weighted avg       0.74      0.75      0.74       705



In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## Initial Insights

- Base model has a accuracy of 76% which is not reliable because of the imbalanced dataset
- TotalCharges needs to be a float/int type (Data Cleaning)
- Need to perform Feature Scaling

## Data Cleaning

In [13]:
telco_data=df.copy()

In [14]:
telco_data.TotalCharges=pd.to_numeric(telco_data.TotalCharges,errors='coerce')

In [15]:
telco_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [16]:
telco_data.loc[telco_data['TotalCharges'].isnull()==True]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [17]:
telco_data.dropna(how='any',inplace=True)

In [18]:
telco_data.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [19]:
telco_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [20]:
telco_data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [21]:
telco_data.tenure.max()

72

In [22]:
bins=[0,12,24,36,48,60,72]

labels=['1-12','13-24','25-36','37-48','40-60','61-72']

telco_data['tenure_bin']=pd.cut(telco_data['tenure'],bins=bins,labels=labels,include_lowest=True)


print(telco_data[['tenure','tenure_bin']].head(10))
print(telco_data['tenure_bin'].value_counts().sort_index())

   tenure tenure_bin
0       1       1-12
1      34      25-36
2       2       1-12
3      45      37-48
4       2       1-12
5       8       1-12
6      22      13-24
7      10       1-12
8      28      25-36
9      62      61-72
tenure_bin
1-12     2175
13-24    1024
25-36     832
37-48     762
40-60     832
61-72    1407
Name: count, dtype: int64


In [23]:
telco_data.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bin
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1-12
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-36
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1-12
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,37-48
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1-12


In [24]:
y=telco_data['Churn']
x=telco_data.drop(columns=['customerID','Churn','tenure'])

print("Shape of X :",x.shape)
print("Shape of y :",y.shape)

Shape of X : (7032, 19)
Shape of y : (7032,)


In [25]:
x

,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,tenure_bin
0,Female,0,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,1-12
1,Male,0,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,25-36
2,Male,0,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1-12
3,Male,0,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,37-48
4,Female,0,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,13-24
7039,Female,0,Yes,Yes,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,61-72
7040,Female,0,Yes,Yes,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,1-12
7041,Male,1,Yes,No,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,1-12


In [26]:
x=pd.get_dummies(x,drop_first=True)
y=telco_data['Churn'].map({'No':0,'Yes':1})

In [27]:
x

,SeniorCitizen,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,...,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_bin_13-24,tenure_bin_25-36,tenure_bin_37-48,tenure_bin_40-60,tenure_bin_61-72
0,0,29.85,29.85,False,True,False,False,True,False,False,...,False,True,False,True,False,False,False,False,False,False
1,0,56.95,1889.50,True,False,False,True,False,False,False,...,False,False,False,False,True,False,True,False,False,False
2,0,53.85,108.15,True,False,False,True,False,False,False,...,False,True,False,False,True,False,False,False,False,False
3,0,42.30,1840.75,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,True,False,False
4,0,70.70,151.65,False,False,False,True,False,False,True,...,False,True,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,84.80,1990.50,True,True,True,True,False,True,False,...,False,True,False,False,True,True,False,False,False,False
7039,0,103.20,7362.90,False,True,True,True,False,True,True,...,False,True,True,False,False,False,False,False,False,True
7040,0,29.60,346.45,False,True,True,False,True,False,False,...,False,True,False,True,False,False,False,False,False,False
7041,1,74.40,306.60,True,True,False,True,False,True,True,...,False,True,False,False,True,False,False,False,False,False


In [28]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

## Feature Scaling

In [29]:
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)

In [30]:
model_dt2.fit(x_train,y_train)
y_pred_dt2=model_dt2.predict(x_test)

In [31]:
print(classification_report(y_test,y_pred_dt2))

              precision    recall  f1-score   support

           0       0.80      0.80      0.80      1012
           1       0.49      0.49      0.49       395

    accuracy                           0.72      1407
   macro avg       0.65      0.65      0.65      1407
weighted avg       0.71      0.72      0.72      1407



## Feature Scaling - MinMaxScaler

In [32]:
x_train_mms=mms.fit_transform(x_train)
x_test_mms=mms.transform(x_test)

In [33]:
model_dt3=DecisionTreeClassifier()
model_dt3.fit(x_train_mms,y_train)

y_pred_dt3=model_dt3.predict(x_test_mms)

In [34]:
print(classification_report(y_test,y_pred_dt3))

              precision    recall  f1-score   support

           0       0.80      0.81      0.81      1012
           1       0.50      0.48      0.49       395

    accuracy                           0.72      1407
   macro avg       0.65      0.64      0.65      1407
weighted avg       0.71      0.72      0.72      1407



## SMOTEENN () [UpSampling + ENN)

In [35]:
x_train_resampled,y_train_resampled=sm.fit_resample(x_train,y_train)

In [36]:
model_dt_smotenn.fit(x_train_resampled,y_train_resampled)
y_pred_dt_smoteenn=model_dt_smotenn.predict(x_test)

In [37]:
print(classification_report(y_test,y_pred_dt_smoteenn))

              precision    recall  f1-score   support

           0       0.88      0.70      0.78      1012
           1       0.50      0.77      0.60       395

    accuracy                           0.72      1407
   macro avg       0.69      0.73      0.69      1407
weighted avg       0.78      0.72      0.73      1407



In [38]:
model_rf_smoteenn.fit(x_train_resampled,y_train_resampled)
y_pred_rf_smoteenn=model_rf_smoteenn.predict(x_test)

In [39]:
print(classification_report(y_test,y_pred_rf_smoteenn))

              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1012
           1       0.52      0.81      0.63       395

    accuracy                           0.73      1407
   macro avg       0.71      0.76      0.71      1407
weighted avg       0.80      0.73      0.75      1407



## XGBoost without SMOTEENN

In [40]:
model_xgb.fit(x_train,y_train)
y_pred_xgb=model_xgb.predict(x_test)

In [41]:
print(classification_report(y_test,y_pred_xgb))

              precision    recall  f1-score   support

           0       0.82      0.87      0.85      1012
           1       0.62      0.52      0.56       395

    accuracy                           0.77      1407
   macro avg       0.72      0.70      0.70      1407
weighted avg       0.76      0.77      0.77      1407



## XGBoost with SMOTEENN

In [42]:
model_xgb_smoteenn.fit(x_train_resampled,y_train_resampled)
y_pred_xgb_smoteenn=model_xgb_smoteenn.predict(x_test)

In [43]:
print(classification_report(y_test,y_pred_xgb_smoteenn))

              precision    recall  f1-score   support

           0       0.90      0.71      0.79      1012
           1       0.52      0.79      0.62       395

    accuracy                           0.73      1407
   macro avg       0.71      0.75      0.71      1407
weighted avg       0.79      0.73      0.75      1407



## XGBoost with SMOTE

In [44]:
print("Before SMOTE: ",Counter(y_train))

x_train_smote,y_train_smote=smote.fit_resample(x_train,y_train)

print("After SMOTE : ",Counter(y_train_smote))

model_xgb_smote.fit(x_train_smote,y_train_smote)

y_pred_xgb_smote=model_xgb_smote.predict(x_test)

Before SMOTE:  Counter({0: 4151, 1: 1474})
After SMOTE :  Counter({1: 4151, 0: 4151})


In [45]:
print(classification_report(y_test,y_pred_xgb_smote))


              precision    recall  f1-score   support

           0       0.82      0.85      0.84      1012
           1       0.59      0.54      0.56       395

    accuracy                           0.76      1407
   macro avg       0.71      0.69      0.70      1407
weighted avg       0.76      0.76      0.76      1407



In [46]:
print("Before ADASYN :" ,Counter(y_train))

x_train_adasyn,y_train_adasyn=adasyn.fit_resample(x_train,y_train)


print("After ADASYN :",Counter(y_train_adasyn))

model_xgb_adasyn.fit(x_train_adasyn,y_train_adasyn)

y_pred_xgb_adasyn=model_xgb_adasyn.predict(x_test)


print("Accuracy Score : ",accuracy_score(y_test,y_pred_xgb_adasyn))
print("\n Classification Report : \n",classification_report(y_test,y_pred_xgb_adasyn))
print("\n Confusion Matrix \n ",confusion_matrix(y_test,y_pred_xgb_adasyn))

Before ADASYN : Counter({0: 4151, 1: 1474})
After ADASYN : Counter({0: 4151, 1: 4025})
Accuracy Score :  0.7711442786069652

 Classification Report : 
               precision    recall  f1-score   support

           0       0.83      0.85      0.84      1012
           1       0.60      0.56      0.58       395

    accuracy                           0.77      1407
   macro avg       0.72      0.71      0.71      1407
weighted avg       0.77      0.77      0.77      1407


 Confusion Matrix 
  [[865 147]
 [175 220]]


In [47]:
scale_pos_weight=(y_train==0).sum()/(y_train==1).sum()
print("Scale_pos_weight:",scale_pos_weight)

model_xgb_weighted=XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)
model_xgb_weighted.fit(x_train,y_train)
y_pred_weighted=model_xgb_weighted.predict(x_test)

print("Accuracy :",accuracy_score(y_test,y_pred_weighted))
print("\n Classification report\n",classification_report(y_test,y_pred_weighted))
print("\n Confusion Matrix\n",confusion_matrix(y_test,y_pred_weighted))

Scale_pos_weight: 2.816146540027137
Accuracy : 0.7526652452025586

 Classification report
               precision    recall  f1-score   support

           0       0.86      0.79      0.82      1012
           1       0.55      0.67      0.60       395

    accuracy                           0.75      1407
   macro avg       0.70      0.73      0.71      1407
weighted avg       0.77      0.75      0.76      1407


 Confusion Matrix
 [[795 217]
 [131 264]]


In [48]:
negative_weight=(y_train==0).sum()
positive_weight=(y_train==1).sum()
weighted_positive=negative_weight/positive_weight
print("Negative calss Count :",negative_weight)
print("positive class count : ",positive_weight)
print("Weighted for Positive class : ",weighted_positive)

model_ada_weighted.fit(x_train,y_train,sample_weight=weighted_positive)
y_pred_ada_weighted=model_ada_weighted.predict(x_test)

print("Adaboost with sample weighting Accuracy",accuracy_score(y_test,y_pred_ada_weighted))
print("\n Classification report \n",classification_report(y_test,y_pred_ada_weighted))
print("\n Confusion Matrix\n",confusion_matrix(y_test,y_pred_ada_weighted))

Negative calss Count : 4151
positive class count :  1474
Weighted for Positive class :  2.816146540027137
Adaboost with sample weighting Accuracy 0.7917555081734187

 Classification report 
               precision    recall  f1-score   support

           0       0.82      0.91      0.86      1012
           1       0.68      0.50      0.57       395

    accuracy                           0.79      1407
   macro avg       0.75      0.70      0.72      1407
weighted avg       0.78      0.79      0.78      1407


 Confusion Matrix
 [[918  94]
 [199 196]]


## **Hyper Parameter Optimization**

In [49]:
param_grid = {
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300, 400],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
    'scale_pos_weight': [1, scale_pos_weight]  # try both balanced and weighted
}

xgb=XGBClassifier(random_state=42,evalmetric='logloss')
search=RandomizedSearchCV(
    xgb,param_grid,n_iter=50,cv=5,scoring='f1',random_state=42,n_jobs=-1
)
search.fit(x_train,y_train)
print("Best Parameters :",search.best_params_)
print("Best CV F1 score : ",search.best_score_)


best_model=search.best_estimator_
y_pred_best=best_model.predict(x_test)

print("\n Test Classifcation Report\n",classification_report(y_test,y_pred_best))

Best Parameters : {'subsample': 0.8, 'scale_pos_weight': 2.816146540027137, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.7}
Best CV F1 score :  0.632053983645472

 Test Classifcation Report
               precision    recall  f1-score   support

           0       0.91      0.73      0.81      1012
           1       0.54      0.81      0.64       395

    accuracy                           0.75      1407
   macro avg       0.72      0.77      0.73      1407
weighted avg       0.80      0.75      0.76      1407



In [50]:
with open('../models/best_xgboost_model.pkl','wb') as f:
    pickle.dump(best_model,f)

print("Best model successfully saved as 'best_xgboost_model.pkl'")

Best model successfully saved as 'best_xgboost_model.pkl'


## Trying out few more techniques

In [51]:
model_rf.fit(x_train,y_train)
y_pred_rf=model_rf.predict(x_test)
print(classification_report(y_test,y_pred_rf))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1012
           1       0.53      0.81      0.64       395

    accuracy                           0.74      1407
   macro avg       0.72      0.76      0.72      1407
weighted avg       0.80      0.74      0.76      1407



In [52]:
model_cat.fit(x_train,y_train)
y_pred_cat=model_cat.predict(x_test)
print(classification_report(y_test,y_pred_cat))

              precision    recall  f1-score   support

           0       0.89      0.76      0.82      1012
           1       0.55      0.75      0.64       395

    accuracy                           0.76      1407
   macro avg       0.72      0.76      0.73      1407
weighted avg       0.79      0.76      0.77      1407



In [53]:
model_lgb.fit(x_train,y_train)
y_pred_lgb=model_lgb.predict(x_test)
print(classification_report(y_test,y_pred_lgb))

[LightGBM] [Info] Number of positive: 1474, number of negative: 4151
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000391 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 606
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.262044 -> initscore=-1.035369
[LightGBM] [Info] Start training from score -1.035369
              precision    recall  f1-score   support

           0       0.83      0.91      0.86      1012
           1       0.68      0.51      0.58       395

    accuracy                           0.80      1407
   macro avg       0.75      0.71      0.72      1407
weighted avg       0.79      0.80      0.79      1407



## **Optuna fine tuning**

In [62]:
def objective(trial):
    params={
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),  # L1 regularization
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2), # L2 regularization
        'scale_pos_weight': trial.suggest_categorical('scale_pos_weight', [1, scale_pos_weight]),
        'random_state': 42,
        'eval_metric': 'logloss'

    }
    model=XGBClassifier(**params)

    f1_scorer=make_scorer(f1_score,average='binary',pos_label=1)
    score=cross_val_score(model,x_train,y_train,cv=5,scoring=f1_scorer,n_jobs=-1).mean()

    return score

study=optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=100)

print("Best parameters ",study.best_params)
print("Best cv f1 score :",study.best_value)


best_model_optuna=XGBClassifier(**study.best_params)
best_model_optuna.fit(x_train,y_train)
y_pred_optuna=best_model.predict(x_test)
print(classification_report(y_test,y_pred_optuna))

[I 2026-05-04 20:58:29,387] A new study created in memory with name: no-name-1a0652cf-bda3-4695-81f9-1db07a594983
[I 2026-05-04 20:58:31,253] Trial 0 finished with value: 0.5456829539473332 and parameters: {'max_depth': 4, 'learning_rate': 0.20000304808614897, 'n_estimators': 618, 'subsample': 0.6870452224542146, 'colsample_bytree': 0.9070564228735644, 'min_child_weight': 2, 'gamma': 0.17220058220556939, 'reg_alpha': 0.9339736229590284, 'reg_lambda': 0.9560013591999641, 'scale_pos_weight': 1}. Best is trial 0 with value: 0.5456829539473332.
[I 2026-05-04 20:58:32,821] Trial 1 finished with value: 0.5779653155638124 and parameters: {'max_depth': 3, 'learning_rate': 0.02399102259068059, 'n_estimators': 827, 'subsample': 0.8591526852441509, 'colsample_bytree': 0.9562751195756963, 'min_child_weight': 10, 'gamma': 0.14178831942917935, 'reg_alpha': 0.4687714514897324, 'reg_lambda': 0.7583437774553758, 'scale_pos_weight': 1}. Best is trial 1 with value: 0.5779653155638124.
[I 2026-05-04 20:58

Best parameters  {'max_depth': 9, 'learning_rate': 0.017601904149510496, 'n_estimators': 206, 'subsample': 0.6637760177725072, 'colsample_bytree': 0.6996214940676014, 'min_child_weight': 8, 'gamma': 0.30091308360613694, 'reg_alpha': 0.5737905349766731, 'reg_lambda': 1.972143014060167, 'scale_pos_weight': 2.816146540027137}
Best cv f1 score : 0.6370406933428618
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1012
           1       0.54      0.81      0.64       395

    accuracy                           0.75      1407
   macro avg       0.72      0.77      0.73      1407
weighted avg       0.80      0.75      0.76      1407



In [63]:
with open('../models/best_optuna_model.pkl','wb') as f:
    pickle.dump(best_model_optuna,f)

print("Best model successfully saved as 'best_optuna_model.pkl")

Best model successfully saved as 'best_optuna_model.pkl


In [64]:
with open('../models/ada_boost_model.pkl','wb')as f:
    pickle.dump(model_ada_weighted,f)

print("Best model successfully saved as 'ada_boost_model.pkl' ")

Best model successfully saved as 'ada_boost_model.pkl' 
